# LeetCode 3492 — Maximum Containers on a Ship bằng Python

Link đề: https://leetcode.com/problems/maximum-containers-on-a-ship/ · Độ khó: **Easy** · Chủ đề: Math

Skeleton LeetCode:

```python
class Solution:
    def maxContainers(self, n: int, w: int, maxWeight: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: boong tàu là lưới `n x n` ô, mỗi ô chứa tối đa **một** container nặng đúng `w`; tổng trọng lượng không được vượt `maxWeight`; tìm **số container nhiều nhất**.
2. Nhận ra bài toán bị chặn bởi **hai giới hạn độc lập**: giới hạn **chỗ** (`n * n` ô) và giới hạn **tải** (`maxWeight // w` container), đáp án là cái **nhỏ hơn**.
3. Dùng **phép chia nguyên** `//` (làm tròn xuống) đúng chỗ và hiểu vì sao không dùng `/` hay làm tròn lên.
4. Viết 4 cách: mô phỏng xếp từng container, công thức `min`, tìm kiếm nhị phân theo đáp án, và xếp theo từng hàng; đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: bỏ quên giới hạn chỗ, bỏ quên giới hạn tải, dùng `n` thay `n * n`, và dùng chia thực/làm tròn lên.

## 2. Đề bài & ví dụ

Cho số nguyên dương `n`: boong tàu là lưới `n x n` ô. **Mỗi ô** chứa được **một** container có trọng lượng **đúng bằng `w`**. Con tàu có **tải trọng tối đa** `maxWeight`: tổng trọng lượng các container đã xếp **không được vượt quá** `maxWeight`.

Trả về **số container tối đa** có thể xếp lên tàu.

| Ví dụ | n, w, maxWeight | Output | Giải thích |
|---|---|---|---|
| 1 | `2, 3, 15` | 4 | boong có `2*2 = 4` ô; tải cho phép `15 // 3 = 5` container; lấy `min(4, 5) = 4` |
| 2 | `3, 5, 20` | 4 | boong có `9` ô; tải cho phép `20 // 5 = 4`; lấy `min(9, 4) = 4` |

Constraints: `1 <= n <= 1000`; `1 <= w <= 1000`; `1 <= maxWeight <= 10^9`.

In [1]:
n, w, maxWeight = 2, 3, 15
cells = n * n
by_weight = maxWeight // w
print("số ô trên boong =", cells)
print("số container tối đa theo tải =", by_weight)
print("đáp án =", min(cells, by_weight))

số ô trên boong = 4
số container tối đa theo tải = 5
đáp án = 4


## 3. Kiến thức nền

### 3.1. Diện tích lưới `n x n`

Lưới `n` hàng, `n` cột có `n * n` ô (`n ** 2`). Với `n = 1000` đó là `1_000_000` ô. Mỗi ô một container nên **số container tối đa theo chỗ** là `n * n`.

In [2]:
for n in (1, 2, 3, 1000):
    print(n, "x", n, "=", n * n, "ô")

1 x 1 = 1 ô
2 x 2 = 4 ô
3 x 3 = 9 ô
1000 x 1000 = 1000000 ô


### 3.2. Chia nguyên `//` và chia thực `/`

`a // b` là **thương nguyên làm tròn xuống** (với số dương): `15 // 3 = 5`, `10 // 3 = 3`. `a / b` là **chia thực** ra số thập phân (`float`): `10 / 3 = 3.333...`. Số container phải là **số nguyên** nên ta dùng `//`: đặt `c` container tốn `c * w`, cần `c * w <= maxWeight`, tức `c <= maxWeight / w`, và vì `c` nguyên nên `c <= maxWeight // w` (làm tròn **xuống**).

In [3]:
print(15 // 3, 10 // 3, 10 / 3, type(10 // 3), type(10 / 3))
print(2999 // 1000, 3000 // 1000)

5 3 3.3333333333333335 <class 'int'> <class 'float'>
2 3


### 3.3. `min` của hai giới hạn

Khi một đại lượng bị **chặn bởi nhiều giới hạn**, đáp án là **giới hạn chặt nhất** (nhỏ nhất): `min(a, b)`. Ở đây không thể vượt số ô và cũng không thể vượt số container mà tải cho phép.

In [4]:
limits = [n * n for n in (2,)] + [15 // 3]
print(limits, min(limits))
print(min(9, 4), min(4, 5))

[4, 5] 4
4 4


### 3.4. Tìm kiếm nhị phân theo đáp án (binary search on answer)

Hàm `feasible(c)` = "xếp được `c` container không?" là **đơn điệu**: nếu xếp được `c` thì xếp được mọi `c' < c`. Khi một điều kiện đơn điệu như vậy, ta nhị phân tìm `c` lớn nhất có `feasible(c)` đúng trong `[0, n*n]`. Với bài này `feasible(c)` là `c <= n*n and c * w <= maxWeight`.

In [5]:
def feasible(c, n, w, maxWeight):
    return c <= n * n and c * w <= maxWeight

print([feasible(c, 2, 3, 15) for c in range(0, 7)])

[True, True, True, True, True, False, False]


### 3.5. Số nguyên lớn và kiểu dữ liệu

`maxWeight` tới `10^9`, `n * n` tới `10^6`, `c * w` tới `10^6 * 10^3 = 10^9`. Trong C++/Java `n * n` và `c * w` vẫn vừa `int` nhưng `1000 * 1000 * 1000` cần cẩn thận (có thể `10^9 < 2^31 - 1 = 2.147 * 10^9`). Python không tràn số nên khỏi lo, nhưng hãy biết giới hạn.

In [6]:
print(1000 * 1000 * 1000, 2**31 - 1, 1000 * 1000 * 1000 <= 2**31 - 1)

1000000000 2147483647 True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Xếp từng container một và dừng khi hết chỗ **hoặc** quá tải. Với `n = 3`, `w = 5`, `maxWeight = 20`:

```
xếp 1: tải 5  <= 20, chỗ 1 <= 9  -> được
xếp 2: tải 10 <= 20, chỗ 2 <= 9  -> được
xếp 3: tải 15 <= 20, chỗ 3 <= 9  -> được
xếp 4: tải 20 <= 20, chỗ 4 <= 9  -> được
xếp 5: tải 25 >  20              -> dừng
```

Đáp án `4`: **tải** là điểm nghẽn (chỗ còn thừa). Ở ví dụ 1 (`n = 2`, `w = 3`, `maxWeight = 15`) thì **chỗ** là điểm nghẽn: sau `4` container hết ô dù tải còn đủ.

## 5. Cách 1 — Brute Force / baseline

Mô phỏng: `count = 0`; trong khi còn **chỗ** (`count < n * n`) và thêm một container vẫn **không quá tải** (`(count + 1) * w <= maxWeight`) thì `count += 1`.

Nút thắt: vòng lặp chạy tối đa `n * n` lần, tới `10^6` với `n = 1000` — vẫn chạy được trong Python nhưng chậm hơn rất nhiều so với `O(1)` và hoàn toàn không cần thiết.

In [7]:
class SolutionSimulate:
    def maxContainers(self, n, w, maxWeight):
        count = 0
        while count < n * n and (count + 1) * w <= maxWeight:
            count += 1
        return count

print(SolutionSimulate().maxContainers(2, 3, 15))
print(SolutionSimulate().maxContainers(3, 5, 20))

4
4


**Độ phức tạp:** Time `O(min(n^2, maxWeight / w))`, Space `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** hai giới hạn không ảnh hưởng nhau, nên ta **tính riêng từng cái** rồi lấy nhỏ hơn:

- giới hạn theo **chỗ**: `n * n`;
- giới hạn theo **tải**: `maxWeight // w`.

**Quy tắc "nếu… thì…":**

- Nếu `maxWeight // w >= n * n` -> đầy boong, đáp án `n * n`.
- Ngược lại -> đáp án `maxWeight // w` (tải hết trước khi hết chỗ).

Invariant: một số `c` hợp lệ **khi và chỉ khi** `c <= n * n` **và** `c * w <= maxWeight`, tức `c <= min(n * n, maxWeight // w)`; số `c` lớn nhất hợp lệ chính là `min(...)`.

In [8]:
def formula(n, w, maxWeight):
    return min(n * n, maxWeight // w)

print(formula(2, 3, 15), formula(3, 5, 20), formula(1000, 1, 10**9), formula(1, 1000, 999))

4 4 1000000 0


## 7. Các cách tiếp theo

Ta có 4 cách: (1) mô phỏng từng container; (2) công thức `min(n * n, maxWeight // w)`; (3) nhị phân theo đáp án; (4) xếp theo từng hàng.

### Cách 2 — Công thức `O(1)`

Một dòng: `min(n * n, maxWeight // w)`.

In [9]:
class SolutionFormula:
    def maxContainers(self, n, w, maxWeight):
        return min(n * n, maxWeight // w)

print(SolutionFormula().maxContainers(2, 3, 15))
print(SolutionFormula().maxContainers(3, 5, 20))

4
4


Độ phức tạp Cách 2: Time `O(1)`, Space `O(1)`.

### Cách 3 — Nhị phân theo đáp án

Tìm `c` lớn nhất trong `[0, n * n]` thoả `c * w <= maxWeight`. Vì `feasible` đơn điệu: `lo = 0` (luôn đúng), `hi = n * n`; lặp lấy `mid = (lo + hi + 1) // 2` (làm tròn **lên** để tránh lặp vô hạn); nếu `mid` khả thi thì `lo = mid` còn không `hi = mid - 1`.

In [10]:
class SolutionBinary:
    def maxContainers(self, n, w, maxWeight):
        lo, hi = 0, n * n
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if mid * w <= maxWeight:
                lo = mid
            else:
                hi = mid - 1
        return lo

print(SolutionBinary().maxContainers(2, 3, 15))
print(SolutionBinary().maxContainers(3, 5, 20))

4
4


Độ phức tạp Cách 3: Time `O(log(n^2)) = O(log n)`, Space `O(1)`. Không cần thiết cho bài này nhưng là kỹ thuật tổng quát khi `feasible` phức tạp hơn.

### Cách 4 — Xếp theo từng hàng

Mỗi hàng có `n` ô. Tính `fit = maxWeight // w` (số container tối đa theo tải). Với mỗi hàng, xếp `min(n, fit còn lại)` container rồi trừ khỏi `fit`; dừng khi `fit` hết. Vòng lặp chạy tối đa `n` lần (thay vì `n * n`).

In [11]:
class SolutionRows:
    def maxContainers(self, n, w, maxWeight):
        remaining = maxWeight // w        # số container còn được phép theo tải
        placed = 0
        for _ in range(n):                # n hàng, mỗi hàng n ô
            if remaining <= 0:
                break
            take = min(n, remaining)
            placed += take
            remaining -= take
        return placed

print(SolutionRows().maxContainers(2, 3, 15))
print(SolutionRows().maxContainers(3, 5, 20))

4
4


Độ phức tạp Cách 4: Time `O(n)`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 4 (theo hàng) với `n = 3`, `w = 5`, `maxWeight = 20` (`remaining = 20 // 5 = 4`):

| hàng | `remaining` trước | `take = min(3, remaining)` | `placed` | `remaining` sau |
|---|---|---|---|---|
| 1 | 4 | 3 | 3 | 1 |
| 2 | 1 | 1 | 4 | 0 |
| 3 | 0 | (dừng, hết tải) | 4 | 0 |

Đáp án `4`.

In [12]:
def trace_rows(n, w, maxWeight):
    remaining = maxWeight // w
    print(f"số container tối đa theo tải = {maxWeight} // {w} = {remaining}")
    placed = 0
    for row in range(1, n + 1):
        if remaining <= 0:
            print(f"hàng {row}: hết tải, dừng")
            break
        take = min(n, remaining)
        placed += take
        remaining -= take
        print(f"hàng {row}: xếp {take} | placed={placed} remaining={remaining}")
    return placed

print("đáp án =", trace_rows(3, 5, 20))
print("đáp án =", trace_rows(2, 3, 15))

số container tối đa theo tải = 20 // 5 = 4
hàng 1: xếp 3 | placed=3 remaining=1
hàng 2: xếp 1 | placed=4 remaining=0
hàng 3: hết tải, dừng
đáp án = 4
số container tối đa theo tải = 15 // 3 = 5
hàng 1: xếp 2 | placed=2 remaining=3
hàng 2: xếp 2 | placed=4 remaining=1
đáp án = 4


## 9. Code nộp LeetCode

In [13]:
from typing import List

class Solution:
    def maxContainers(self, n: int, w: int, maxWeight: int) -> int:
        return min(n * n, maxWeight // w)

print(Solution().maxContainers(2, 3, 15), Solution().maxContainers(3, 5, 20))

4 4


## 10. Vì sao thuật toán đúng

- **Điều kiện hợp lệ:** xếp `c` container là hợp lệ khi và chỉ khi (a) `c <= n * n` (mỗi ô một container) và (b) `c * w <= maxWeight` (không quá tải). (Container đồng nhất `w` nên mọi cách chọn ô cho cùng tổng tải; vị trí ô không ảnh hưởng.)
- **Đổi dạng (b):** `c * w <= maxWeight` <=> `c <= maxWeight / w`; vì `c` là số nguyên nên tương đương `c <= floor(maxWeight / w) = maxWeight // w`.
- **Số lớn nhất:** `c` hợp lệ nhất nằm đúng ở cận trên nhỏ hơn trong hai cận: `min(n * n, maxWeight // w)`. Cận này đạt được vì `c = min(...)` thoả cả hai điều kiện.
- **Cách 1** đếm lên cho tới khi vi phạm một trong hai điều kiện -> cũng ra `c` lớn nhất. **Cách 3** nhị phân đúng vì `feasible` đơn điệu. **Cách 4** phân phối `maxWeight // w` container vào các hàng `n` ô, tổng không vượt `n * n` nên `placed = min(remaining ban đầu, n * n)`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng từng container | `O(min(n^2, maxWeight / w))` | `O(1)` |
| 2. Công thức `min` | `O(1)` | `O(1)` |
| 3. Nhị phân theo đáp án | `O(log n)` | `O(1)` |
| 4. Xếp theo từng hàng | `O(n)` | `O(1)` |

Cách 2 tối ưu. Cách 1 là chuẩn đối chiếu (chỉ dùng với `n` nhỏ trong test để chạy nhanh).

## 12. Bộ test

Test 1: ví dụ đề và các edge case (`n = 1`, `w` quá nặng -> `0`, tải vừa khít, giới hạn chỗ, giới hạn tải, biên lớn).

In [14]:
impls = [SolutionSimulate(), SolutionFormula(), SolutionBinary(), SolutionRows(), Solution()]
tests = [
    (2, 3, 15, 4),
    (3, 5, 20, 4),
    (1, 1, 1, 1),                      # tối thiểu mọi tham số
    (1, 1000, 999, 0),                 # w > maxWeight -> 0 container
    (1, 5, 5, 1),                      # vừa khít tải
    (4, 2, 32, 16),                    # tải đủ chính xác cho hết boong (16 * 2 = 32)
    (4, 2, 31, 15),                    # thiếu 1 đơn vị tải -> 15 (31 // 2)
    (4, 2, 1000, 16),                  # chỗ là điểm nghẽn
    (3, 1, 8, 8),                      # tải là điểm nghẽn, 8 < 9
    (3, 1, 9, 9),                      # đúng bằng số ô
    (1000, 1, 10**9, 10**6),           # biên: n * n = 10^6 là điểm nghẽn
    (1000, 1000, 10**9, 10**6),        # 10^9 // 1000 = 10^6 = n * n
    (1000, 1000, 10**9 - 1, 999999),   # thiếu 1 -> 999999
]
for n, w, mw, exp in tests:
    assert 1 <= n <= 1000 and 1 <= w <= 1000 and 1 <= mw <= 10**9
    for im in impls:
        if isinstance(im, SolutionSimulate) and n * n > 10**4:
            continue                    # brute chạy tối đa 10^6 vòng, bỏ qua ca rất lớn để test chạy nhanh
        got = im.maxContainers(n, w, mw)
        assert got == exp, (type(im).__name__, n, w, mw, got, exp)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 13


Test 2: ngẫu nhiên có seed, đúng constraints. Với tham số nhỏ đối chiếu **cả mô phỏng**; với tham số lớn đối chiếu công thức, nhị phân, theo hàng.

In [15]:
import random
random.seed(3492)

for t in range(2000):
    n = random.randint(1, 30)
    w = random.randint(1, 50)
    mw = random.randint(1, 3000)
    assert 1 <= n <= 1000 and 1 <= w <= 1000 and 1 <= mw <= 10**9
    exp = impls[0].maxContainers(n, w, mw)
    for im in impls:
        assert im.maxContainers(n, w, mw) == exp, (type(im).__name__, n, w, mw)
print("2000 test nhỏ (có mô phỏng) OK")

for t in range(2000):
    n = random.randint(1, 1000)
    w = random.randint(1, 1000)
    mw = random.randint(1, 10**9)
    exp = SolutionFormula().maxContainers(n, w, mw)
    for im in impls[2:]:
        assert im.maxContainers(n, w, mw) == exp, (type(im).__name__, n, w, mw)
print("2000 test lớn OK")

2000 test nhỏ (có mô phỏng) OK


2000 test lớn OK


Test 3: tính chất — đáp án `<= n * n`; `answer * w <= maxWeight`; `(answer + 1) * w > maxWeight` **hoặc** `answer == n * n` (tối đại); đơn điệu theo `maxWeight` (tải lớn hơn không làm đáp án giảm); giảm theo `w`.

In [16]:
random.seed(92)
for _ in range(1000):
    n = random.randint(1, 1000)
    w = random.randint(1, 1000)
    mw = random.randint(1, 10**9)
    ans = Solution().maxContainers(n, w, mw)
    assert 0 <= ans <= n * n
    assert ans * w <= mw
    assert ans == n * n or (ans + 1) * w > mw
    assert Solution().maxContainers(n, w, mw + random.randint(0, 1000)) >= ans
    assert Solution().maxContainers(n, w + random.randint(0, 5), mw) <= ans
print("tính chất (<= n*n, không quá tải, tối đại, đơn điệu): OK")

tính chất (<= n*n, không quá tải, tối đại, đơn điệu): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chỉ tính theo tải, quên giới hạn chỗ

In [17]:
def wrong_only_weight(n, w, maxWeight):
    return maxWeight // w                       # SAI: có thể vượt số ô trên boong

print("sai :", wrong_only_weight(2, 3, 15))
print("đúng:", Solution().maxContainers(2, 3, 15))
assert wrong_only_weight(2, 3, 15) != Solution().maxContainers(2, 3, 15)

sai : 5
đúng: 4


Vì sao sai: tải cho phép `5` container nhưng boong `2 x 2` chỉ có `4` ô. **Sửa:** `min(n * n, maxWeight // w)`.

### Lỗi 2: chỉ tính theo chỗ, quên giới hạn tải

In [18]:
def wrong_only_cells(n, w, maxWeight):
    return n * n                                # SAI: bỏ qua trọng lượng

print("sai :", wrong_only_cells(3, 5, 20))
print("đúng:", Solution().maxContainers(3, 5, 20))
assert wrong_only_cells(3, 5, 20) != Solution().maxContainers(3, 5, 20)

sai : 9
đúng: 4


Vì sao sai: `9` container nặng `5` là `45 > 20`, quá tải. Phải cân cả hai giới hạn.

### Lỗi 3: dùng `n` thay vì `n * n` (nhầm "lưới `n x n`" với "một hàng `n`")

In [19]:
def wrong_n_not_square(n, w, maxWeight):
    return min(n, maxWeight // w)               # SAI: boong có n * n ô, không phải n ô

print("sai :", wrong_n_not_square(3, 1, 100))
print("đúng:", Solution().maxContainers(3, 1, 100))
assert wrong_n_not_square(3, 1, 100) != Solution().maxContainers(3, 1, 100)

sai : 3
đúng: 9


Vì sao sai: `n x n` có `n * n` ô; với `n = 3` là `9`, không phải `3`. **Sửa:** `n * n`.

### Lỗi 4: dùng chia thực `/` hoặc làm tròn lên

In [20]:
import math

def wrong_float(n, w, maxWeight):
    return min(n * n, maxWeight / w)            # SAI: ra số thực 3.333..., không phải số container nguyên

def wrong_ceil(n, w, maxWeight):
    return min(n * n, math.ceil(maxWeight / w)) # SAI: làm tròn LÊN nên quá tải

print("sai (/)   :", wrong_float(2, 3, 10))
print("sai (ceil):", wrong_ceil(2, 3, 10))
print("đúng      :", Solution().maxContainers(2, 3, 10))
assert wrong_float(2, 3, 10) != Solution().maxContainers(2, 3, 10)
assert wrong_ceil(2, 3, 10) != Solution().maxContainers(2, 3, 10)

sai (/)   : 3.3333333333333335
sai (ceil): 4
đúng      : 3


Vì sao sai: `10 / 3 = 3.33` không phải số container hợp lệ; làm tròn lên cho `4` container nặng `12 > 10`, quá tải. Phải làm tròn **xuống** bằng `//`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tối đa bao nhiêu vật, chịu **hai giới hạn**" -> đáp án là `min` của hai giới hạn tính riêng.
- Vật đồng nhất (trọng lượng cố định) -> **không cần mô phỏng**, chỉ cần chia: `maxWeight // w`.
- Số lượng nguyên -> dùng `//` (làm tròn xuống); `ceil` dành cho bài hỏi "**ít nhất** bao nhiêu".
- Lưới `n x n` -> `n * n` ô.
- Bài đơn điệu ("xếp được `c` thì xếp được `c - 1`") luôn có thể **nhị phân theo đáp án**.
- Kiểm tra biên: `w > maxWeight` -> `0`; `n = 1`; số rất lớn (`10^9`).

## 15. Biến thể của bài

1. Các container có **trọng lượng khác nhau** `weights[]`: chọn nhiều container nhất mà tổng `<= maxWeight` -> **sắp xếp tăng dần rồi lấy từ nhỏ nhất** (tham lam), và vẫn bị chặn bởi `n * n`.
2. Mỗi ô chứa **nhiều** container (`k` container mỗi ô): chỗ là `n * n * k`.
3. Boong **chữ nhật** `rows x cols`: chỗ là `rows * cols`.
4. Thêm **chi phí** mỗi container, tối đa hoá giá trị với tải giới hạn (knapsack).
5. Tải hỏi **ít nhất bao nhiêu chuyến tàu** để chở `N` container: `ceil(N / min(n*n, maxWeight // w))`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | xếp từng container tới khi vi phạm | `O(min(n^2, W/w))` | `O(1)` | hiểu đề, chuẩn đối chiếu |
| 2. Công thức `min` | `min(n*n, maxWeight // w)` | `O(1)` | `O(1)` | **chuẩn, nên nộp** |
| 3. Nhị phân theo đáp án | `feasible(c)` đơn điệu | `O(log n)` | `O(1)` | khi điều kiện phức tạp hơn |
| 4. Theo từng hàng | trừ dần số container còn được phép | `O(n)` | `O(1)` | mô phỏng nhanh |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `maxContainers(4, 7, 100)` bằng bao nhiêu?

**Bài 2 (tính tay):** `n = 5`, `w = 2`, `maxWeight = 61`: giới hạn theo chỗ, giới hạn theo tải, đáp án?

**Bài 3 (điền TODO):** hoàn thành `capacity_by_weight(w, maxWeight)` trả về số container tối đa theo tải.

**Bài 4 (tự viết lại):** viết `bottleneck(n, w, maxWeight)` trả về chuỗi `"space"`, `"weight"` hoặc `"both"` cho biết giới hạn nào chặn đáp án.

**Bài 5 (mini challenge):** các container có trọng lượng khác nhau `weights`; viết `max_containers_varied(n, weights, maxWeight)` trả về số container nhiều nhất (tối đa `n * n`) có tổng `<= maxWeight`.

In [21]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().maxContainers(4, 7, 100))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def capacity_by_weight(w, maxWeight):
    # TODO: số container nguyên tối đa mà tổng trọng lượng không vượt maxWeight
    return 0

14


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** tính hai con số: số ô và số container tải cho phép; lấy số nhỏ hơn.

**Mức 2 (vừa):** Bài 3: dùng `//`. Bài 4: so sánh `n * n` với `maxWeight // w` (bằng nhau -> `"both"`). Bài 5: sắp xếp `weights`, cộng dần từ nhỏ nhất cho tới khi vượt `maxWeight` hoặc đủ `n * n`.

**Mức 3 (gần đáp án):** Bài 1: chỗ `16`, tải `100 // 7 = 14` -> `14`. Bài 2: chỗ `25`, tải `61 // 2 = 30` -> `25` (chỗ chặn).

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1: min(16, 100 // 7 = 14) = 14
assert Solution().maxContainers(4, 7, 100) == 14

# Bài 2: min(25, 30) = 25
assert Solution().maxContainers(5, 2, 61) == 25

# Bài 3
def capacity_by_weight(w, maxWeight):
    return maxWeight // w
assert capacity_by_weight(3, 15) == 5
assert capacity_by_weight(1000, 999) == 0

# Bài 4
def bottleneck(n, w, maxWeight):
    a, b = n * n, maxWeight // w
    if a == b:
        return "both"
    return "space" if a < b else "weight"
assert bottleneck(2, 3, 15) == "space"
assert bottleneck(3, 5, 20) == "weight"
assert bottleneck(2, 1, 4) == "both"

# Bài 5
def max_containers_varied(n, weights, maxWeight):
    total, count = 0, 0
    for x in sorted(weights):
        if count == n * n or total + x > maxWeight:
            break
        total += x
        count += 1
    return count
assert max_containers_varied(2, [5, 1, 3, 2], 6) == 3        # 1 + 2 + 3
assert max_containers_varied(1, [1, 1, 1], 100) == 1         # chỉ 1 ô
assert max_containers_varied(3, [4, 4], 3) == 0
assert max_containers_varied(2, [3] * 4, 12) == Solution().maxContainers(2, 3, 12)
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "tối đa số vật chịu hai giới hạn":**

```python
by_space = n * n                 # giới hạn chỗ
by_weight = maxWeight // w       # giới hạn tải (làm tròn xuống)
return min(by_space, by_weight)
```

**Checklist trước khi submit:**

- [ ] Có đủ **hai** giới hạn: chỗ `n * n` và tải `maxWeight // w`?
- [ ] Dùng `n * n` (không phải `n`)?
- [ ] Dùng `//` (chia nguyên, làm tròn xuống), không dùng `/` hay `ceil`?
- [ ] Lấy `min` của hai giới hạn?
- [ ] Đã thử `w > maxWeight` (đáp án `0`), `n = 1`, các giá trị lớn `10^9`?
- [ ] Chữ ký đúng `maxContainers(self, n: int, w: int, maxWeight: int) -> int`?